# ERA5 Multi-Year Cutout Download — British Columbia

**Author:** Md Eliasinul Islam  
**Affiliation:** Delta E+ Lab, Simon Fraser University  
**Purpose:** Download ERA5 climate cutouts for BC across multiple weather years using the `ERA5Cutout` class from the RESource framework.

---

## Design rationale

`ERA5Cutout.get_era5_cutout()` reads its temporal horizon from `config['cutout']['snapshots']`.  
Rather than writing one config YAML per year (brittle), this notebook:

1. Instantiates `ERA5Cutout` once from the base BC config (loads boundaries, resolves paths).
2. For each target year, calls `era5_bc.get_year_snapshot(year)` — which reads `region_mapping[BC]['timezone_convert']` and converts local midnight Jan 1 → UTC — then patches `cutout_config['snapshots']` in-place.
3. Calls `get_era5_cutout()` per year with **retry logic** and **inter-year throttling** to respect CDS API queue limits.
4. Validates coverage and file integrity after all downloads.

**Timezone note:**  
Snapshot UTC strings are derived automatically from `timezone_convert: Etc/GMT+8` in the BC region config.  
POSIX `Etc/GMT+8` = UTC−8 (PST), so local midnight Jan 1 maps to `YYYY-01-01 08:00:00 UTC`.  
End is `YYYY+1-01-01 07:00:00 UTC` (last local hour of the year) — no overlap between adjacent years.

**Prerequisite:** A valid `~/.cdsapirc` with your CDS API key must exist on this machine.

---
## 0 · Configuration  
Edit **only this cell** before running the notebook.

In [ ]:
from pathlib import Path

SCENARIO="baseline"
WEATHER_YEAR= str(2023)
# ── Region & resource ─────────────────────────────────────────────────────
CONFIG_FILE_PATH   = Path("config/CAN_baseline.yaml")   # adjust to your BC config location
REGION_SHORT_CODE  = "BC"
RESOURCE_TYPE      = "wind"   # 'wind' | 'solar'  — affects config lookups only

# ── Weather years to download ─────────────────────────────────────────────
# Each year produces an independent NetCDF cutout file.
# ERA5 availability: 1940-present (CDS v2).
WEATHER_YEARS = list(range(2000, 2025))  # adjust as needed; e.g. range(2015, 2024) for 2015-2023

# ── Download control ─────────────────────────────────────────────────────
OVERWRITE_EXISTING  = False  # True: re-download even if .nc already exists on disk

# ── CDS API throttling ────────────────────────────────────────────────────
# CDS rejects simultaneous queued jobs from the same user.
# INTER_YEAR_DELAY_S: pause between consecutive year downloads.
# MAX_RETRIES: attempts per year before marking as failed.
# RETRY_BACKOFF_S: base wait before first retry; doubles on each subsequent retry.
INTER_YEAR_DELAY_S  = 60    # seconds (set to 0 to disable; >=60 recommended)
MAX_RETRIES         = 3     # per year
RETRY_BACKOFF_S     = 120   # seconds (first retry: 120 s, second: 240 s, ...)

print(f"Config   : {CONFIG_FILE_PATH}  (exists: {CONFIG_FILE_PATH.exists()})")
print(f"Region   : {REGION_SHORT_CODE}")
print(f"Resource : {RESOURCE_TYPE}")
print(f"Years    : {WEATHER_YEARS}")
print(f"Inter-year delay  : {INTER_YEAR_DELAY_S} s")
print(f"Retries / backoff : {MAX_RETRIES} x {RETRY_BACKOFF_S} s (doubling)")

---
## 1 · Imports

In [ ]:
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

import RES.utility as utils
from RES.era5_cutout import ERA5Cutout

# ── RESource package on path ───────────────────────────────────────────────
CODEBASE_ROOT = Path("../codebase").resolve()
if str(CODEBASE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODEBASE_ROOT))


print(f"RES package loaded from: {CODEBASE_ROOT}")

### 1b · tqdm backend fix

Force `tqdm` to use the **standard text-based** progress bar instead of the notebook ipywidgets backend.

**Why:** `atlite` internally calls `tqdm.notebook.tqdm` for download progress. In Jupyter environments where `ipywidgets` comm is not fully initialised, this fails with `LookupError: shell_parent ContextVar`. Replacing the class reference routes all progress output to plain stdout — no effect on download logic.

In [ ]:
import tqdm.notebook
import tqdm.std

tqdm.notebook.tqdm = tqdm.std.tqdm
print("tqdm backend: notebook widget -> text (ipywidgets comm bypass active)")

---
## 2 · Instantiate ERA5Cutout (once)

The class constructor loads GADM boundaries and resolves config paths.  
We do this once and reuse the instance across all years.  
The cell below also previews what `get_year_snapshot()` produces for the first target year.

In [ ]:
utils.print_module_title("ERA5 Multi-Year Download — British Columbia")

era5_bc = ERA5Cutout(
    config_file_path  = CONFIG_FILE_PATH,
    region_short_code = REGION_SHORT_CODE,
    resource_type     = RESOURCE_TYPE,
)

print("\nInstance ready.")
print(f"  Country       : {era5_bc.country}")
print(f"  Region code   : {era5_bc.region_short_code}")
print(f"  Timezone      : {era5_bc.region_timezone}")
print(f"  Cutout root   : {era5_bc.cutout_config.get('root')}")
print(f"  Grid (dx, dy) : ({era5_bc.cutout_config['dx']}deg, {era5_bc.cutout_config['dy']}deg)")

# Preview snapshot strings for each target year
print("\nSnapshot preview (derived from timezone_convert):")
print(f"  {'Year':<6}  {'start (UTC)':<22}  {'end (UTC)'}")
print(f"  {'----':<6}  {'-------------------':<22}  {'-------------------'}")
for yr in WEATHER_YEARS:
    s, e = era5_bc.get_snapshot(year=yr)
    print(f"  {yr:<6}  {s:<22}  {e}")

---
## 4 · Pre-download audit

In [ ]:
# 1. Initialize an empty list to store each year's DataFrame
        
# Generate the list of dataframes dynamically, filtering out warnings if needed
audit_dfs = [era5_bc.audit_cutout(year) for year in WEATHER_YEARS]

# Combine them immediately
combined_audit_df = pd.concat(audit_dfs, ignore_index=True)

# 3. Apply style mapping and format for notebook display
display(
    combined_audit_df.style
    .map(  # Updated from .applymap to .map for Pandas 2.x compliance
        lambda v: "color: #2e7d32; font-weight: bold" if v is True else
                  "color: #c62828; font-weight: bold" if v is False else "",
        subset=["exists"]
    )
    .format(
        # Hardened to handle floats, ints, and ignore NaNs/Strings gracefully
        {"size_MB": lambda x: f"{float(x):.1f} MB" if isinstance(x, (int, float)) and not pd.isna(x) else "--"}
    )
)

for year in WEATHER_YEARS:
    if year < 1940 or year > 2024:
        print(f"Warning: Year {year} is outside the typical ERA5 range (1940-2024). Check availability before downloading.")
    audit_df = era5_bc.audit_cutout(
                                year)

years_to_download = [
    row["year"] for _, row in combined_audit_df.iterrows()
    if (not row["exists"]) or OVERWRITE_EXISTING
]
years_skipped = [y for y in WEATHER_YEARS if y not in years_to_download]

print(f"\nYears to download : {years_to_download}")
print(f"Years to skip     : {years_skipped}  (already on disk; OVERWRITE_EXISTING={OVERWRITE_EXISTING})")

---
## 5 · Multi-year ERA5 download loop

For each target year:
1. `patch_cutout_year()` calls `get_year_snapshot(year)` to compute timezone-correct UTC bounds and updates the instance in-place.
2. `get_era5_cutout()` submits monthly CDS API requests via atlite and writes to NetCDF.
3. On **CDS rate-limit rejections** (`400: Number queued requests temporarily limited`), the loop waits with exponential backoff and retries up to `MAX_RETRIES`.
4. `INTER_YEAR_DELAY_S` between years prevents simultaneous job queuing.

> **Runtime:** BC at 0.25 deg is ~200-400 MB yr-1. CDS typically yields ~15-30 min per year.

In [ ]:
_CDS_RATE_LIMIT_PHRASES = ("temporarily limited", "number queued", "has been rejected")

def _is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(phrase in msg for phrase in _CDS_RATE_LIMIT_PHRASES)


download_log = []

for i, year in enumerate(years_to_download):

    utils.print_banner(f" Downloading ERA5 cutout -- BC -- {year} ")

    t0      = time.time()
    status  = "failed"
    err_msg = None

    for attempt in range(1, MAX_RETRIES + 1):

        if attempt > 1:
            wait_s = RETRY_BACKOFF_S * (2 ** (attempt - 2))
            utils.print_update(
                level=3,
                message=f"Retry {attempt}/{MAX_RETRIES} for {year} -- waiting {wait_s} s..."
            )
            time.sleep(wait_s)

        try:
            cutout, region_boundary = era5_bc.get_era5_cutout(weather_year=year)
            status  = "success"
            err_msg = None
            break

        except Exception as exc:
            err_msg       = str(exc)
            is_rate_limit = _is_rate_limit_error(exc)
            utils.print_update(
                level=3,
                message=f"Attempt {attempt}/{MAX_RETRIES} FAILED for {year} "
                        f"({'rate-limit' if is_rate_limit else 'other'}): {err_msg[:120]}",
                alert=True
            )
            if not is_rate_limit or attempt == MAX_RETRIES:
                traceback.print_exc()
                break

    elapsed = round(time.time() - t0, 1)
    download_log.append({
        "year"        : year,
        "status"      : status,
        "elapsed_min" : round(elapsed / 60, 1),
        "cutout_path" : str(era5_bc.get_cutout_path(year)),
        "error"       : err_msg,
    })
    utils.print_update(level=3, message=f"Year {year} | status={status} | elapsed={elapsed/60:.1f} min")

    if INTER_YEAR_DELAY_S > 0 and i < len(years_to_download) - 1:
        utils.print_update(level=4, message=f"Inter-year pause: {INTER_YEAR_DELAY_S} s...")
        time.sleep(INTER_YEAR_DELAY_S)


print("\n--- Download loop complete ---")
pd.DataFrame(download_log)

---
## 6 · Post-download validation

In [ ]:
import calendar

validation_rows = []
for _, row in combined_audit_df.iterrows():
    yr   = int(row["year"])
    path = Path(row["path"])
    meta = era5_bc.validate_cutout_nc(path)

    if meta is None:
        validation_rows.append({"year": yr, "status": "missing"})
    elif "error" in meta:
        validation_rows.append({"year": yr, "status": "corrupt", "error": meta["error"]})
    else:
        expected    = 8784 if calendar.isleap(yr) else 8760
        coverage_ok = (meta["n_timesteps"] == expected)
        validation_rows.append({
            "year"        : yr,
            "status"      : "complete" if coverage_ok else "partial",
            "n_timesteps" : meta["n_timesteps"],
            "expected"    : expected,
            "t_start"     : meta["t_start"],
            "t_end"       : meta["t_end"],
            "size_MB"     : meta["size_MB"],
            "variables"   : ", ".join(meta["variables"]),
        })

display(pd.DataFrame(validation_rows))

---
## 7 · Summary manifest

In [ ]:
from datetime import datetime

print("=" * 65)
print("  ERA5 MULTI-YEAR DOWNLOAD MANIFEST -- BC")
print("=" * 65)
print(f"  Timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Config     : {CONFIG_FILE_PATH}")
print(f"  Region     : {REGION_SHORT_CODE}")
print(f"  Timezone   : {era5_bc.region_timezone}")
print(f"  Years      : {WEATHER_YEARS}")
print(f"  Grid       : dx={era5_bc.cutout_config['dx']}deg, dy={era5_bc.cutout_config['dy']}deg")
print()

if download_log:
    for entry in download_log:
        flag = "v" if entry["status"] == "success" else "x"
        s, e = era5_bc.get_snapshot(year=entry["year"])
        print(f"  [{flag}] {entry['year']}  {s} -> {e}  | {entry['elapsed_min']:.1f} min")
        if entry["error"]:
            print(f"       error: {entry['error'][:120]}")
else:
    print("  No years downloaded in this run (all already on disk).")

if years_skipped:
    for yr in years_skipped:
        row = combined_audit_df[combined_audit_df["year"] == yr].iloc[0]
        print(f"  [-] {yr}  skipped | {row['size_MB']} MB | {row['path']}")

print("=" * 65)

---
## Appendix A · Troubleshooting

| Symptom | Root cause | Fix |
|---------|-----------|-----|
| `LookupError: shell_parent ContextVar` | ipywidgets comm not initialised | Ensure cell 1b ran before the loop |
| `400: Number queued requests temporarily limited` | CDS API concurrent job limit | Handled by retry + `INTER_YEAR_DELAY_S`; increase delay if persists |
| `ZoneInfoNotFoundError` | Bad IANA identifier in `timezone_convert` | Check config; valid forms: `Etc/GMT+8`, `America/Vancouver` |
| Snapshot looks wrong (off by sign) | POSIX Etc/GMT+N sign inversion | `Etc/GMT+8` = UTC-8; cell 3 preview table shows derived UTC times |
| `partial` in validation | CDS request dropped mid-transfer | Set `OVERWRITE_EXISTING=True` and re-run |
| Very slow (>60 min/yr) | CDS queue depth | Run overnight; `INTER_YEAR_DELAY_S` already helps |

---
## Appendix B · Extending to multiple regions

Re-instantiate `ERA5Cutout` per region (boundaries differ). `patch_cutout_year()` and `get_year_snapshot()` work unchanged because they read `timezone_convert` from the instance's own `region_short_code`.

```python
for region_code in ["BC", "AB", "SK"]:
    era5_region = ERA5Cutout(
        config_file_path  = Path(f"config/config_{region_code}.yaml"),
        region_short_code = region_code,
        resource_type     = RESOURCE_TYPE,
    )
    for year in WEATHER_YEARS:
        patch_cutout_year(era5_region, year)
        era5_region.get_era5_cutout()
```